# Notebook 11 — B1-M3: Full-panel exit gate (checkpoint-only)

**Project B1 — target reframing, confirmatory full-panel run.** B1-M2's slice
ablation (`notebooks/10_b1_target_ablation.ipynb`) was **provisional**
(METHODOLOGY §11): its 2018-start slice had **zero `qe_bull` OOS bars**, so it
could only speak to `covid` + `rate_cycle`, and its pinned carry-forward flagged
one candidate — `directional_21d` (edge in `covid` only). This notebook renders
the **binary B1 verdict** on the full **33-symbol × ~22-year panel** (2004→2026),
which restores all three required regimes.

It is **checkpoint-only** (METHODOLOGY §7): it loads the per-target checkpoints
written by `scripts/run_b1_arms.py` and scores them through `b1_gate_report` (the
source of truth, METHODOLOGY §2). **It never fits a model and never reloads
prices** — every fit-derived artifact (GBM/ARIMA predictions, the directional
Sharpe-arm return series) lives in the checkpoints; the deterministic baselines
(EWMA, climatology) were computed by the runner and checkpointed too.

### The gate (`b1_gate_report`, conjunction of three stages)
1. **Materiality** — every criterion in `spec.materiality` met in >= `min_pass`(=2)
   of the required regimes `(qe_bull, covid, rate_cycle)`.
2. **Significance** — paired stationary-block-bootstrap (21-day blocks) 90% CI of
   the gated-metric delta **excludes 0** in >= 1 required regime.
3. **Deflation** — deflated Sharpe > 0 (directional) or skill-z analog > 0 (T1/T2),
   with the deflation N read from the trial-count ledger (+ this matrix's pinned
   self-comparison count).

**Gate outcome -> action.** Any target clearing all three stages in >= 2 required
regimes is a B-cycle artifact (Phase-5 Trigger 1) and a Project-C candidate. If
*no* target clears, the verdict is "no extractable edge from this feature set on
any of the four targets," which trips the B3/B4 conditional skip paths
(`.claude/prds/b1-target-reframing.prd.md` Sequencing notes).

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, roc_auc_score

from quant.backtest.metrics import compute_metrics
from quant.backtest.regime_metrics import DEFAULT_REGIMES_REQUIRED, b1_gate_report
from quant.backtest.regimes import DateRangeDetector, tag_regimes
from quant.backtest.statistics import (
    bootstrap_metric_delta_ci,
    bootstrap_sharpe_delta_ci,
    deflated_sharpe_ratio,
    forecast_skill_z,
)
from quant.features.targets import TARGET_CATALOG
from quant.ledger import load_ledger

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

# Pinned config (must match scripts/run_b1_arms.py; METHODOLOGY 1)
CHECKPOINT_ROOT = Path("../data/b1")
REGIMES = DEFAULT_REGIMES_REQUIRED            # (qe_bull, covid, rate_cycle)
MIN_PASS = 2
BOOT = dict(block_len=21, n_boot=1000, ci=0.90, seed=0)   # 21-day block convention
N_SELF_COMPARISONS = len(TARGET_CATALOG) * len(REGIMES)   # 4x3 = 12, pinned

# Deflation N for the gate: cumulative trials from OTHER milestones + this
# matrix's own self-comparisons. Excluding B1-M3's own ledger entries makes this
# notebook deterministic on re-execution AFTER the post-gate ledger logging
# (otherwise re-running would double-count and shift the DSR).
_entries = load_ledger()
LEDGER_N_OTHER = sum(e.n_comparisons for e in _entries if e.milestone != "B1-M3")
LEDGER_N_SELF = LEDGER_N_OTHER + N_SELF_COMPARISONS
print(f"targets: {list(TARGET_CATALOG)}")
print(f"required regimes: {REGIMES}  min_pass: {MIN_PASS}")
print(f"deflation N: {LEDGER_N_SELF} (other-milestone {LEDGER_N_OTHER} + self {N_SELF_COMPARISONS})")

## 1 - Load the four target checkpoints

Each `data/b1/{target}/` carries `metadata.json`, `predictions.parquet`
(date-indexed `symbol, y_true, gbm_pred` + per-target baseline columns), and -
for the directional targets - `returns_gbm.parquet` / `returns_arima.parquet`
(the tradeable-Sharpe-arm OOS return series).

In [ ]:
def _load_returns(target_dir, arm):
    p = target_dir / f"returns_{arm}.parquet"
    if not p.exists():
        return pd.Series(dtype=float)
    s = pd.read_parquet(p)[f"{arm}_returns"]
    s.index = pd.DatetimeIndex(s.index)
    return s.dropna().sort_index()


checkpoints = {}
missing = []
for tid in TARGET_CATALOG:
    tdir = CHECKPOINT_ROOT / tid
    meta_p = tdir / "metadata.json"
    if not meta_p.exists():
        missing.append(tid)
        continue
    frame = pd.read_parquet(tdir / "predictions.parquet")
    frame.index = pd.DatetimeIndex(frame.index)
    checkpoints[tid] = dict(
        meta=json.loads(meta_p.read_text()),
        frame=frame,
        gbm_ret=_load_returns(tdir, "gbm"),
        arima_ret=_load_returns(tdir, "arima"),
    )

assert not missing, (
    f"missing checkpoints for {missing} - run scripts/run_b1_arms.py --target <t> "
    "for each before executing this notebook (it never fits a model)."
)
print(f"loaded checkpoints for: {list(checkpoints)}")

## 2 - Frozen-config audit (METHODOLOGY 8 - invariant parity from metadata)

Pin every invariant the verdict depends on from each arm's `metadata.json`: the
config hash, git SHA, panel breadth, OOS span, label horizon, the drawdown base
rate, and the runner's invariant-parity audit text (recorded *before* the arms
ran).

In [ ]:
rows = []
for tid, ck in checkpoints.items():
    m = ck["meta"]
    br = m["drawdown_base_rate"]
    br_clean = (None if (br is None or (isinstance(br, float) and np.isnan(br)))
                else round(float(br), 3))
    rows.append({
        "target": tid,
        "horizon": m["label_horizon"],
        "n_symbols": m["n_symbols_in_panel"],
        "n_oos_rows": m["n_oos_rows"],
        "n_oos_dates": m["n_oos_dates"],
        "oos_start": str(m["oos_start"])[:10],
        "oos_end": str(m["oos_end"])[:10],
        "base_rate": br_clean,
        "config_hash": m["config_hash"][:12],
    })
audit = pd.DataFrame(rows).set_index("target")
display(audit)

git_shas = {ck["meta"]["git_sha"] for ck in checkpoints.values()}
print(f"\ngit SHA(s) across arms: {git_shas}")
print("invariant-parity audit (recorded BEFORE the arms ran):")
print(" ", next(iter(checkpoints.values()))["meta"]["invariant_parity_audit"])

n_syms = {ck["meta"]["n_symbols_in_panel"] for ck in checkpoints.values()}
print(f"\nsymbols per arm: {n_syms}")
assert min(n_syms) >= 25, "full-panel run should cover the ~33-symbol universe"


## 3 - Regime composition: all three required regimes now populated

The slice could not test `qe_bull`. Tag each target's OOS calendar and confirm
the full panel restores all three required regimes (the whole point of the
confirmatory run).

In [ ]:
def regime_series(frame):
    return tag_regimes(pd.DatetimeIndex(frame.index), DateRangeDetector())


comp = {}
for tid, ck in checkpoints.items():
    rs = regime_series(ck["frame"])
    comp[tid] = rs[rs.isin(REGIMES)].value_counts().reindex(REGIMES).fillna(0).astype(int)
comp_df = pd.DataFrame(comp).T
display(comp_df)
empty = [r for r in REGIMES if (comp_df[r] == 0).any()]
print(f"\nrequired regimes with zero rows on SOME target: {empty or '(none - all populated)'}")

## 4 - Per-target scoring

The same scoring logic as nb10, but every input comes from a checkpoint (no fit,
no price reload). Each target yields per-regime materiality metrics, the gated
metric's bootstrap significance CI, and a deflation verdict, then `b1_gate_report`
renders the gate.

In [ ]:
def _auc(y, p):
    y = np.asarray(y, dtype=float)
    if len(np.unique(y[~np.isnan(y)])) < 2:
        raise ValueError("single class - AUC undefined")
    return float(roc_auc_score(y, p))


def _mae(y, p):
    return float(mean_absolute_error(y, p))


def _auc_cs(arr):
    return _auc(arr[:, 0], arr[:, 1])


def _mae_cs(arr):
    return _mae(arr[:, 0], arr[:, 1])


gate_results = {}
score_detail = {}

In [ ]:
# T1: drawdown_21d (AUC, skill-z)
spec = TARGET_CATALOG["drawdown_21d"]
ck = checkpoints["drawdown_21d"]
frame = ck["frame"]
rs = regime_series(frame).to_numpy()
y_all = frame["y_true"].to_numpy(dtype=float)
gbm = frame["gbm_pred"].to_numpy(dtype=float)
vol = frame["vol_proxy"].to_numpy(dtype=float)
base_rate = float(ck["meta"]["drawdown_base_rate"])

pr_metrics, sig_ci = {}, {}
for rg in REGIMES:
    m = rs == rg
    if not m.any():
        continue
    yv = y_all[m]
    gbm_auc = _auc(yv, gbm[m]) if len(np.unique(yv)) > 1 else np.nan
    try:
        vol_auc = _auc(yv, vol[m])
    except ValueError:
        vol_auc = np.nan
    clim_auc = 0.5
    better = np.nanmax([clim_auc, vol_auc])
    pr_metrics[rg] = {"auc": {"variant": gbm_auc, "baseline": float(better)}}
    use_vol = (not np.isnan(vol_auc)) and (vol_auc >= clim_auc)
    base_score = vol[m] if use_vol else np.full(m.sum(), 0.5)
    variant = np.column_stack([yv, gbm[m]])
    baseline = np.column_stack([yv, base_score])
    try:
        sig_ci[rg] = bootstrap_metric_delta_ci(variant, baseline, _auc_cs, **BOOT)
    except ValueError:
        sig_ci[rg] = None

skill = (base_rate - y_all) ** 2 - (gbm - y_all) ** 2  # Brier improvement vs climatology
skillz = forecast_skill_z(skill)
gate_results["drawdown_21d"] = b1_gate_report(
    spec, pr_metrics, sig_ci, skillz.passed,
    regimes_required=REGIMES, min_pass=MIN_PASS, deflation_detail=str(skillz),
)
score_detail["drawdown_21d"] = dict(base_rate=base_rate, skillz=skillz)
print(f"T1 base rate P[DD>5%]={base_rate:.3f}  skill-z={skillz.z:.3f} (pass={skillz.passed})")
print("T1 per-regime AUC (variant/baseline):",
      {r: (round(v['auc']['variant'], 3), round(v['auc']['baseline'], 3)) for r, v in pr_metrics.items()})

In [ ]:
# T2: realized_vol_21d (MAE, skill-z)
spec = TARGET_CATALOG["realized_vol_21d"]
ck = checkpoints["realized_vol_21d"]
frame = ck["frame"]
rs = regime_series(frame).to_numpy()
y_all = frame["y_true"].to_numpy(dtype=float)
gbm = frame["gbm_pred"].to_numpy(dtype=float)
ewma = frame["ewma_pred"].to_numpy(dtype=float)
arima = frame["arima_pred"].to_numpy(dtype=float)

e_ok, a_ok = ~np.isnan(ewma), ~np.isnan(arima)
mae_ewma = _mae(y_all[e_ok], ewma[e_ok]) if e_ok.any() else np.inf
mae_arima = _mae(y_all[a_ok], arima[a_ok]) if a_ok.any() else np.inf
best_base = ewma if mae_ewma <= mae_arima else arima
print(f"T2 aggregate MAE: EWMA={mae_ewma:.4f}  ARIMA-on-logvol={mae_arima:.4f}  GBM={_mae(y_all, gbm):.4f}")

pr_metrics, sig_ci = {}, {}
for rg in REGIMES:
    m = rs == rg
    if not m.any():
        continue
    yv = y_all[m]
    gbm_mae = _mae(yv, gbm[m])
    em, am = ewma[m], arima[m]
    em_ok, am_ok = ~np.isnan(em), ~np.isnan(am)
    mae_e = _mae(yv[em_ok], em[em_ok]) if em_ok.any() else np.inf
    mae_a = _mae(yv[am_ok], am[am_ok]) if am_ok.any() else np.inf
    better = min(mae_e, mae_a)
    pr_metrics[rg] = {"mae": {"variant": gbm_mae, "baseline": float(better)}}
    base_pred = em if mae_e <= mae_a else am
    ok = ~np.isnan(base_pred)
    variant = np.column_stack([yv[ok], gbm[m][ok]])
    baseline = np.column_stack([yv[ok], base_pred[ok]])
    try:
        sig_ci[rg] = bootstrap_metric_delta_ci(variant, baseline, _mae_cs, **BOOT)
    except ValueError:
        sig_ci[rg] = None

ok = ~np.isnan(best_base)
skill = np.abs(y_all[ok] - best_base[ok]) - np.abs(y_all[ok] - gbm[ok])
skillz = forecast_skill_z(skill)
gate_results["realized_vol_21d"] = b1_gate_report(
    spec, pr_metrics, sig_ci, skillz.passed,
    regimes_required=REGIMES, min_pass=MIN_PASS, deflation_detail=str(skillz),
)
score_detail["realized_vol_21d"] = dict(skillz=skillz, mae_ewma=mae_ewma, mae_arima=mae_arima)
print(f"T2 skill-z (vs better baseline)={skillz.z:.3f} (pass={skillz.passed})")
print("T2 per-regime MAE (variant/baseline):",
      {r: (round(v['mae']['variant'], 4), round(v['mae']['baseline'], 4)) for r, v in pr_metrics.items()})

In [ ]:
# T3 / T4: directional (AUC + tradeable Sharpe, DSR)
def score_directional(tid):
    spec = TARGET_CATALOG[tid]
    ck = checkpoints[tid]
    frame = ck["frame"]
    rs = regime_series(frame).to_numpy()
    y_all = frame["y_true"].to_numpy(dtype=float)
    gbm = frame["gbm_pred"].to_numpy(dtype=float)
    arima_score = frame["arima_pred"].to_numpy(dtype=float)
    gbm_ret, arima_ret = ck["gbm_ret"], ck["arima_ret"]
    gbm_reg = tag_regimes(gbm_ret.index, DateRangeDetector()) if len(gbm_ret) else pd.Series(dtype=object)
    ar_reg = tag_regimes(arima_ret.index, DateRangeDetector()) if len(arima_ret) else pd.Series(dtype=object)

    pr_metrics, sig_ci = {}, {}
    for rg in REGIMES:
        m = rs == rg
        if not m.any():
            continue
        yv = y_all[m]
        gbm_auc = _auc(yv, gbm[m]) if len(np.unique(yv)) > 1 else np.nan
        try:
            arima_auc = _auc(yv, arima_score[m])
        except ValueError:
            arima_auc = np.nan
        better_auc = np.nanmax([0.5, arima_auc])
        gbm_r = gbm_ret[(gbm_reg == rg).to_numpy()] if len(gbm_ret) else gbm_ret
        ar_r = arima_ret[(ar_reg == rg).to_numpy()] if len(arima_ret) else arima_ret
        gbm_sharpe = compute_metrics(gbm_r)["sharpe"] if len(gbm_r) else np.nan
        ar_sharpe = compute_metrics(ar_r)["sharpe"] if len(ar_r) else np.nan
        pr_metrics[rg] = {
            "auc": {"variant": gbm_auc, "baseline": float(better_auc)},
            "sharpe": {"variant": float(gbm_sharpe), "baseline": float(ar_sharpe)},
        }
        try:
            sig_ci[rg] = (bootstrap_sharpe_delta_ci(gbm_r, ar_r, **BOOT)
                          if (len(gbm_r) and len(ar_r)) else None)
        except ValueError:
            sig_ci[rg] = None

    try:
        dsr = deflated_sharpe_ratio(gbm_ret, LEDGER_N_SELF)
        dsr_passed = dsr.passed
    except ValueError:
        dsr, dsr_passed = None, False
    gate = b1_gate_report(
        spec, pr_metrics, sig_ci, dsr_passed,
        regimes_required=REGIMES, min_pass=MIN_PASS,
        deflation_detail=(str(dsr) if dsr else "n/a"),
    )
    return gate, dict(dsr=dsr, gbm_ret=gbm_ret, arima_ret=arima_ret)


for tid in ("directional_5d", "directional_21d"):
    g, det = score_directional(tid)
    gate_results[tid] = g
    score_detail[tid] = det
    dsr = det["dsr"]
    gs = compute_metrics(det['gbm_ret'])['sharpe'] if len(det['gbm_ret']) else float('nan')
    as_ = compute_metrics(det['arima_ret'])['sharpe'] if len(det['arima_ret']) else float('nan')
    msg = f"  {tid}: GBM Sharpe={gs:.3f}  ARIMA Sharpe={as_:.3f}  "
    msg += f"DSR={dsr.dsr:.3f} (pass={dsr.passed})" if dsr else "DSR n/a"
    print(msg)

## 5 - Gate verdicts (`b1_gate_report`, the source of truth)

In [ ]:
def verdict_row(tid):
    g = gate_results[tid]
    return {
        "target": tid,
        "materiality": f"{g['material_pass_count']}/{len(REGIMES)} (need {MIN_PASS})",
        "materiality_passed": g["materiality_passed"],
        "significance_passed": g["significance_passed"],
        "deflation_passed": g["deflation_passed"],
        "GATE": g["gate_passed"],
    }

verdict = pd.DataFrame([verdict_row(t) for t in TARGET_CATALOG]).set_index("target")
display(verdict)
cleared = [t for t in TARGET_CATALOG if gate_results[t]["gate_passed"]]
print(f"\nTargets clearing the full-panel gate: {cleared or '(none)'}")

### Per-regime materiality + significance detail (every gated criterion)

In [ ]:
for tid in TARGET_CATALOG:
    g = gate_results[tid]
    print(f"\n=== {tid} ===  gate_passed={g['gate_passed']}  "
          f"deflation={g['deflation_passed']} :: {g['deflation_detail']}")
    for rg in REGIMES:
        pr = g["per_regime"].get(rg, {})
        if not pr:
            print(f"  {rg:11s} (no data)")
            continue
        crit = ", ".join(
            f"{c['metric']}: val={None if c['value'] is None else round(c['value'], 4)} "
            f"(thr {c['threshold']}, met={c['met']})"
            for c in pr.get("criteria", [])
        )
        ci = pr.get("significance_ci")
        ci_s = "n/a" if ci is None else f"[{ci[0]:.4f}, {ci[1]:.4f}] excl0={pr['ci_excludes_zero']}"
        print(f"  {rg:11s} mat_met={str(pr['materiality_met']):5s} {crit} | CI {ci_s}")

## 6 - Borda composite across the three required regimes (METHODOLOGY 10)

Targets carry heterogeneous metrics, so we rank them per regime by a unitless
**materiality margin** - the gated metric's delta divided by its pinned threshold
(1.0 = exactly at the bar) - then Borda-sum the ranks across the required regimes.
This never cherry-picks the winning regime.

In [ ]:
def materiality_margin(tid, rg):
    g = gate_results[tid]
    pr = g["per_regime"].get(rg, {})
    margins = []
    for c in pr.get("criteria", []):
        v = c["value"]
        if v is None or (isinstance(v, float) and np.isnan(v)):
            margins.append(-np.inf)
        else:
            margins.append(v / c["threshold"] if c["threshold"] else v)
    return min(margins) if margins else -np.inf


def _regime_has_data(rg):
    for t in TARGET_CATALOG:
        for c in gate_results[t]["per_regime"].get(rg, {}).get("criteria", []):
            v = c["value"]
            if v is not None and not (isinstance(v, float) and np.isnan(v)):
                return True
    return False

populated_req = [r for r in REGIMES if _regime_has_data(r)]
margin_tbl = pd.DataFrame(
    {rg: {t: materiality_margin(t, rg) for t in TARGET_CATALOG} for rg in populated_req}
)
borda = pd.DataFrame(index=list(TARGET_CATALOG))
for rg in populated_req:
    borda[rg] = margin_tbl[rg].replace(-np.inf, np.nan).rank(method="average", na_option="bottom")
borda["borda_total"] = borda[populated_req].sum(axis=1)
borda = borda.sort_values("borda_total", ascending=False)
print("Materiality margins (gated delta / threshold; -inf = undefined/NaN):")
display(margin_tbl.round(3))
print("\nBorda composite (higher = stronger across regimes):")
display(borda.round(2))

## 7 - Binary B1 verdict + conditional skip-path determination

**This is the confirmatory full-panel verdict** (METHODOLOGY 11) - unlike the
slice, it is not provisional. A pass on any target satisfies Phase-5 Trigger 1 and
nominates that (target, model) tuple for Project C. A fail on all four is the
pre-committed "no extractable edge from this feature set" outcome, tripping the
B3/B4 conditional skip paths.

In [ ]:
cleared = [t for t in TARGET_CATALOG if gate_results[t]["gate_passed"]]
print("B1-M3 FULL-PANEL EXIT-GATE VERDICT")
print("=" * 64)
print(f"deflation N: {LEDGER_N_SELF}  (other-milestone {LEDGER_N_OTHER} + self {N_SELF_COMPARISONS})")
print(f"Targets clearing the gate (>=2 regimes + sig + deflation): {cleared or '(none)'}")
print(f"Borda full-panel leader: {borda.index[0]} (total {borda['borda_total'].iloc[0]:.1f})")
print("=" * 64)
if cleared:
    print(f"VERDICT: EDGE FOUND on {cleared}. Phase-5 Trigger 1 satisfied; these "
          "(target, model) tuples are Project-C deployment candidates.")
    print("Skip paths: B3/B4 stay blocked/low-priority unless separately motivated.")
else:
    print("VERDICT: NO EXTRACTABLE EDGE from the M6 feature set on any of the four "
          "targets, on the full 33-symbol panel across all three required regimes.")
    print("This trips the pre-committed conditional skip paths (PRD Sequencing):")
    print("  -> draft B3 (options-implied alt data); B4 (universe shift) follows only")
    print("     if B1+B3 both surface no edge. A failed B1 cannot be revived by")
    print("     alternative justification without a new PRD (METHODOLOGY 5).")

## 8 - Ledger logging commands (run post-gate; METHODOLOGY 12)

The runner writes one append-only ledger entry per arm with the verdict the gate
just determined (the verdict is not knowable at fit time). Logging is idempotent
by `config_hash` and re-logs from the existing checkpoint **without re-fitting**.
The cell below prints the exact commands; they are run from the repo root.

In [ ]:
print("# Per-arm ledger logging (verdict from the gate above)")
for tid in TARGET_CATALOG:
    g = gate_results[tid]
    v = "gate_passed" if g["gate_passed"] else "gate_failed"
    print(
        f".venv/bin/python scripts/run_b1_arms.py --target {tid} --log-ledger "
        f"--ledger-verdict {v} --ledger-agent human "
        f'--ledger-notes "B1-M3 full-panel arm; see docs/B1_REPORT.md"'
    )